# Etapa 2 — Tratamento dos dados

**Sumário do trabalho**

1. **Análise exploratória** (`01_analise_exploratoria.ipynb`): base, faltantes, alvo e análise bivariada.
2. **Tratamento dos dados** (`02_tratamento_dados.ipynb`): remoção de vazamento; faltantes (estrutural, exame não pedido, esporádico); codificação das qualitativas; remoção de colunas vazias, constantes e redundantes. ← *este notebook*
3. **Modelos** (`03_modelos.ipynb`): divisão treino/teste 80/20; árvore de decisão podada, regressão logística Lasso (seleciona os atributos) e Random Forest; hiperparâmetros por validação cruzada de 10 *folds*.
4. **Avaliação** (`04_avaliacao.ipynb`): AUC na validação cruzada e no teste, com intervalo de confiança e comparação com o acaso; matriz de confusão; uso clínico (limiar para não perder eventos e curva de decisão); interpretação dos três modelos.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().resolve().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src import utils, preprocessing, features

utils.setup_plotting()

### Matriz de atributos e alvo

Toda a etapa está em `features.build_dataset()`. Nenhum passo usa o alvo, então a matriz é montada antes da divisão treino/teste.

In [ ]:
patients, drugs, data_dictionary = preprocessing.load_data()
variable_groups = preprocessing.get_variable_groups(patients)

X, y, report = features.build_dataset(patients, drugs, variable_groups)

print(f"Matriz: {X.shape[0]} internações x {X.shape[1]} atributos (todos numéricos)")
print(f"Alvo: {y.sum()} eventos ({y.mean():.1%}) de óbito ou readmissão em 6 meses")

### Colunas removidas

Vazamento (desfechos, dados da alta, identificador), vazias, constantes e redundantes com o Glasgow. Os desfechos removidos são os que formam o alvo.

In [ ]:
removed = report["removed"]
display(removed["motivo"].value_counts().to_frame("colunas"))
removed.groupby("motivo")["coluna"].apply(", ".join).to_frame("colunas removidas")

### Faltantes

| Tipo | Tratamento |
|---|---|
| Estrutural: vazio em `respiratory.support.` = sem ventilação | Coluna 0/1 `respiratory.support` |
| Exame não pedido: mais de 20% de faltantes | Um indicador 0/1 por painel; o valor medido é mantido |
| Esporádico: até 20% | Imputação pela mediana no treino (etapa 3) |

Indicadores de exame não pedido (`nao_pedido_pct`: internações sem o exame):

In [ ]:
pd.set_option("display.max_colwidth", 80)
report["panels"]

### Matriz final

In [ ]:
drug_cols = [c for c in X.columns if c.startswith("drug_")]
indicator_cols = report["panels"]["indicador"].tolist()
binary_cols = [c for c in X.columns if X[c].dropna().isin([0, 1]).all()]

pd.Series({
    "Numéricas": X.shape[1] - len(binary_cols),
    "Binárias clínicas e codificadas": len(binary_cols) - len(drug_cols) - len(indicator_cols),
    "Indicadores de exame não pedido": len(indicator_cols),
    "Fármacos": len(drug_cols),
    "Total": X.shape[1],
    "Colunas com valores vazios (imputados pela mediana na etapa 3)": int((X.isna().mean() > 0).sum()),
}).to_frame("colunas")